In [9]:
import html
import pathlib
import re
import zipfile
import xml.etree.ElementTree as ET

def createCleanedText(filePath):
    docxPath = pathlib.Path(filePath)
    ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}

    with zipfile.ZipFile(docxPath, 'r') as docx:
        settingsData = docx.read('word/settings.xml').decode('utf-8')

    root = ET.fromstring(settingsData)

    docVars = root.findall(".//w:docVars/w:docVar", ns)

    package_entries = []
    for docvar in docVars:
        name = docvar.get("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}name")
        if name and name.startswith("DOCDRAFTERPACKAGE_"):
            value = docvar.get("{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val")
            if value:
                package_entries.append((name, value))

    if not package_entries:
        raise ValueError("No DOCDRAFTERPACKAGE_* values were found in word/settings.xml")

    package_entries.sort(key=lambda item: int(item[0].split("_")[-1]) if item[0].split("_")[-1].isdigit() else -1)

    cleaned_parts = []
    for package_name, package_value in package_entries:
        cleaned = package_value.replace("_x00d__x00a_", "\n")
        cleaned = cleaned.replace("_x000d_", "\n")
        cleaned = cleaned.replace("_x000a_", "\n")
        cleaned = cleaned.replace("\r\n", "\n").replace("\r", "\n")
        cleaned = html.unescape(cleaned)

        if cleaned.startswith("<?xml"):
            cleaned = cleaned.split("?>", 1)[1].strip()

        cleaned_parts.append(f"<!-- {package_name} -->\n{cleaned}")

    cleaned = "\n\n".join(cleaned_parts)

    output_path = pathlib.Path('cleaned.txt')
    output_path.write_text(cleaned, encoding='utf-8')

    print("Processed", len(package_entries), "package entries")
    print("Saved cleaned XML to", output_path)
    print(cleaned)


def findIrrelevantVars(docx_path, cleaned_text_path="cleaned.txt", prefixes=None):
    docx_path = pathlib.Path(docx_path)
    cleaned_path = pathlib.Path(cleaned_text_path)

    if not cleaned_path.exists():
        createCleanedText(docx_path)

    ns = {"w": "http://schemas.openxmlformats.org/wordprocessingml/2006/main"}
    prefixes = {prefix.upper() for prefix in (prefixes or ["Q", "A", "F"])}

    with zipfile.ZipFile(docx_path, 'r') as docx:
        document_xml = docx.read('word/document.xml').decode('utf-8', errors='ignore')

    root = ET.fromstring(document_xml)
    field_code_pattern = re.compile(r"(?<![A-Za-z])([A-Za-z])\s*(\d+)(?![A-Za-z0-9])")

    field_codes = []
    for alias in root.findall('.//w:alias', ns):
        text = alias.get('{http://schemas.openxmlformats.org/wordprocessingml/2006/main}val', '') or ''
        for match in field_code_pattern.finditer(text):
            prefix = match.group(1).upper()
            if prefix not in prefixes:
                continue
            number = int(match.group(2))
            field_codes.append({
                "prefix": prefix,
                "number": number,
                "token": f"{prefix}{number}",
                "context": text.strip(),
            })

    cleaned_text = cleaned_path.read_text(encoding='utf-8')
    display_pattern = re.compile(r"<(?:DisplayNum|DisplayNumber)>(\d+)</(?:DisplayNum|DisplayNumber)>", re.IGNORECASE)
    display_numbers = {int(match.group(1)) for match in display_pattern.finditer(cleaned_text)}

    missing_numbers = sorted({item['number'] for item in field_codes if item['number'] not in display_numbers})
    code_numbers = {item['number'] for item in field_codes}
    missing_display_tuples = [("Q", number) for number in sorted(display_numbers) if number not in code_numbers]

    return {
        "field_codes": field_codes,
        "display_numbers": sorted(display_numbers),
        "missing_numbers": missing_numbers,
        "missing_display_tags": sorted(display_numbers),

        "missing_display_tuples": missing_display_tuples,
    }

In [10]:
createCleanedText("Separation Agreement (Employment) (NY).docx")
result = findIrrelevantVars("Separation Agreement (Employment) (NY).docx")
print("field codes found:", len(result["field_codes"]))
print("unique codes:", sorted({(item["prefix"], item["number"]) for item in result["field_codes"]}, key=lambda item: item[1]))
print("display numbers:", result["display_numbers"])
print("missing display tuples:", result["missing_display_tuples"])

Processed 4 package entries
Saved cleaned XML to cleaned.txt
<!-- DOCDRAFTERPACKAGE_0 -->
<Package xmlns:xsd="http://www.w3.org/2001/XMLSchema" xmlns:xsi="http://www.w3.org/2001/XMLSchema-instance">

  <XMLVersion>3</XMLVersion>

  <PortalUrl>https://lexisnexis.documentdrafter.com/</PortalUrl>

  <Version>3</Version>

  <Name>Separation Agreement (Employment) (NY)</Name>

  <Id>c6af4034-c31c-4923-91a1-c8ec4e8c6d17</Id>

  <Templates>

    <Template>

      <Name>2478329_LE_SeparationAgreement_NY.docx</Name>

      <DownloadName>[F20]</DownloadName>

      <Id>673ecb04-096a-453a-9f8f-75aa6dfe02c8</Id>

      <Conditions />

      <Clauses />

      <Calculations />

      <RepeatId />

      <updated>2026-07-28T08:01:32.7265631Z</updated>

      <isDirty>false</isDirty>

    </Template>

  </Templates>

  <Questions>

    <Question>

      <DisplayNumber>1</DisplayNumber>

      <Id>b6e0f75f-ef20-4ac7-8dff-deb6b7658f93</Id>

      <LinkBitQuestionId />

      <LinkBitId />

      <Seque

In [ ]:
import tkinter as tk
from tkinter import filedialog, messagebox

def chooseDocx():
    file_path = filedialog.askopenfilename(title="Select DOCX file", filetypes=[("Word Documents", "*.docx")])
    if file_path:
        docx_path_var.set(file_path)
        output_text.delete("1.0", tk.END)

def runCleanAndFind():
    docx_path = docx_path_var.get().strip()
    if not docx_path:
        messagebox.showwarning("No File Selected", "Please select a .docx file first.")
        return
    try:
        createCleanedText(docx_path)
        result = findIrrelevantVars(docx_path)
        unique_codes = sorted({(item["prefix"], item["number"]) for item in result["field_codes"]}, key=lambda item: item[1])
        output_lines = [
            f"field codes found: {len(result['field_codes'])}\n"
            f"unique codes: {unique_codes}\n"
            f"display numbers: {result['display_numbers']}\n"
            f"missing display tuples: {result['missing_display_tuples']}\n"
        ]
        output_text.delete("1.0", tk.END)
        output_text.insert(tk.END, "\n".join(output_lines))
    except Exception as exc:
        messagebox.showerror("Error", str(exc))

root = tk.Tk()
root.title("Document Drafter Tool")
root.geometry("760x440")

docx_path_var = tk.StringVar()

main_frame = tk.Frame(root, padx=10, pady=10)
main_frame.pack(fill=tk.BOTH, expand=True)

tk.Label(main_frame, text="Selected DOCX:").grid(row=0, column=0, sticky="w")
tk.Entry(main_frame, textvariable=docx_path_var, width=70).grid(row=0, column=1, sticky="we", padx=(5, 0))
tk.Button(main_frame, text="Browse", command=chooseDocx).grid(row=0, column=2, padx=(5, 0))

tk.Button(main_frame, text="Find Irrelevant Variables", command=runCleanAndFind).grid(row=1, column=0, columnspan=3, pady=(10, 0), sticky="we")

tk.Label(main_frame, text="Results:").grid(row=2, column=0, columnspan=3, sticky="w", pady=(10, 0))
output_text = tk.Text(main_frame, height=12, wrap="word")
output_text.grid(row=3, column=0, columnspan=3, sticky="nsew")
scrollbar = tk.Scrollbar(main_frame, command=output_text.yview)
scrollbar.grid(row=3, column=3, sticky="ns")
output_text.config(yscrollcommand=scrollbar.set)

main_frame.columnconfigure(1, weight=1)
main_frame.rowconfigure(3, weight=1)

root.mainloop()

In [2]:
from pathlib import Path
import re
from html import unescape


def build_explanatory_note_dict(file_path="cleaned.txt"):
    path = Path(file_path)
    if not path.exists():
        path = Path.cwd() / file_path

    text = path.read_text(encoding="utf-8", errors="ignore")
    question_blocks = re.findall(r"<Question>(.*?)</Question>", text, re.DOTALL)
    explanatory_notes = {}

    for block in question_blocks:
        display_number_match = re.search(r"<DisplayNumber>(.*?)</DisplayNumber>", block, re.DOTALL)
        if not display_number_match:
            continue

        display_number = display_number_match.group(1).strip()
        explanatory_note_match = re.search(r"<ExplanatoryNote>(.*?)</ExplanatoryNote>", block, re.DOTALL)

        if explanatory_note_match:
            note = explanatory_note_match.group(1).strip()
            if note:
                note = re.sub(r"<[^>]+>", " ", note)
                note = re.sub(r"\s+", " ", note).strip()
                note = unescape(note)
            else:
                note = "NO EXPLANATORY NOTE"
        else:
            note = "NO EXPLANATORY NOTE"

        explanatory_notes[display_number] = note

    return explanatory_notes


explanatory_note_dict = build_explanatory_note_dict()
print(f"Created {len(explanatory_note_dict)} entries.")
print(explanatory_note_dict)

Created 39 entries.
{'1': 'NO EXPLANATORY NOTE', '2': 'NO EXPLANATORY NOTE', '3': 'Drafting Note This form is drafted to satisfy the requirements of the Older Workers Benefit Protection Act of 1990 (OWBPA) for Employees age 40 or older to knowingly and voluntarily waive claims under the Age Discrimination in Employment Act (ADEA), including by providing the Employee with a minimum review period and revocation period, among other requirements. 29 U.S.C. § 626(f)(1). As noted in Drafting Note to Older Workers Benefit Protection Act , these requirements are not needed for younger Employees (who are not protected under the ADEA), but many Employers use the same OWBPA-compliant form separation agreement for all Employees and apply the relevant review period, revocation period, and other terms accordingly. OWBPA requires that Employees 40 and over have at least 21 days to consider an agreement that contains an ADEA waiver when the termination is not part of a group termination, or 45 days if